[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/symmetry/symmetry.ipynb)

# Material Symmetry

This example asks which material responses a symmetric object can have. A crystal's properties must look the same after any of its symmetries, so averaging a measured response over the crystal's symmetry group keeps exactly the part the crystal allows. Heat conduction in a cubic crystal comes out the same in every direction, yet its elastic stiffness keeps a cubic shape; and a crystal can rotate polarized light only if it has no mirror or centre of inversion, unless its mirrors exchange directions with opposite senses.

Each response is modelled as a map: a conductivity from driving fields, read by crossing them, to heat flows along directions, each principal axis reading the field by their pairing, `field & axis`; an elasticity a form on four planes, each reading a bond's stretch, `plane & bond`; a rotary power a form on a direction twice, read by the crystal's principal planes, with an intrinsic, handed value. A symmetry moves a map as it moves anything, `rotor >> conductivity(rotor << Field)`, and averaging the moved maps over a group projects the response onto the invariant ones. The rotary power has a sense, so an improper rotation, an odd versor, reverses it where it leaves the conductivity alone: the reversal comes with the rotary power's type, not with a sign rule.

In the tensor notation of crystal physics, the conductivity is a symmetric second-rank tensor $K_{ij}$, the stiffness the fourth-rank tensor $C_{ijkl}$ and the rotary power the gyration tensor $g_{ij}$, and Neumann's principle asks each to be unchanged by every element $R$ of the point group, $K_{ij} = R_{ia} R_{jb} K_{ab}$, while the gyration tensor carries the determinant, $g_{ij} = \det(R)\, R_{ia} R_{jb}\, g_{ab}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor, NumpyContext, concatenate, stack
from examples.util.animation import save_figure
from examples.mechanics.symmetry import render
from examples.mechanics.symmetry.core import ga, lattice_samples, sphere, unit_fields

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# A driving field, the temperature's fall per unit length, is read by crossing it: a vector. Heat flows,
# bonds and the paths of light are carried by geometry: directions, antivectors. The planes that read
# directions, a strain's and a rotary power's, are vectors too.
Field = ga.gatype.vector()                                                      # Vector+
Plane = ga.gatype.vector()                                                      # Vector+
Direction = ga.gatype.antivector()                                              # Antivector-
# A turn per distance with a sense of its own: an intrinsic scalar.
RotaryPower = Scalar.derive.flipped_orientation                                 # Scalar-
Conductivity = ga.gatype((Direction, Field))                                    # Direction- <- Field+
Elasticity = ga.gatype((Scalar, Plane, Plane, Plane, Plane))                    # Scalar+ <- (Plane+, Plane+, Plane+, Plane+)
Gyration = ga.gatype((RotaryPower, Direction, Direction))                       # RotaryPower- <- (Direction-, Direction-)
# A plane of turning, the turns in it, and the improper rotations, the odd versors.
TurningPlane = ga.gatype.bivector()                                             # Bivector+
Rotor = ga.gatype.rotor()                                                       # Rotor+
ImproperRotation = ga.gatype.odd()                                              # Odd+

## 1. Heat conduction

A measured conductivity, positive, with three different gains along a tilted frame of directions, is averaged over three groups: half turns in the planes `mv.yz` and `mv.xy`, quarter turns in `mv.xy`, and the rotations of a cube. Each panel is the image of the unit ball of driving fields, a heat-flow ellipsoid, with one driving field, dark, drawn at the largest gain, and the heat flow it drives, orange. Half turns align the ellipsoid with the axes; quarter turns in `mv.xy` make it round in that plane; the cube makes it a sphere, and heat then flows along the driving field. The measured conductivity has six independent components, and the three groups leave three, two and one.

In [ ]:
# The measured responses' tilted principal frame, the conductivity's gains along it, and the driving field drawn.
pose = (mv.xy * 0.3 + mv.yz * 0.16).exp()                                      # [] Rotor+
axes = pose >> stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))                  # [axes] Direction-
gains = np.array([4.5, 1.5, 0.6])
driving = (mv.x + mv.y + mv.z).normalized()                                    # [] Field+

In [ ]:
def turns(plane: TurningPlane, order: int) -> Rotor:
    """The cyclic group of `order` equal turns in a unit plane, the identity first."""
    # A rotor turns by twice its angle: element k, `(plane * (np.pi * k / order)).exp()`, turns by
    # `2 * np.pi * k / order`.
    return (plane * (np.pi * np.arange(order) / order)).exp()


def cube_rotations() -> Rotor:
    """The 24 rotations of a cube: each of the six faces turned to the top, then four quarter turns in `mv.xy`."""
    tilt = np.pi / 4 * np.array([0, 1, 2, 3, 0, 0])
    roll = np.pi / 4 * np.array([0, 0, 0, 0, 1, -1])
    faces = (mv.yz * tilt + mv.zx * roll).exp()
    return (faces[:, None] * turns(mv.xy, 4)[None, :]).reshape(-1)


def conductivities(axes: Direction, gains: np.ndarray, groups: list[Rotor]) -> Conductivity:
    """A positive conductivity along principal axes, followed by its average over each group.

    A conductivity maps the driving field, the temperature's fall, to heat flow; in an anisotropic material the
    flow need not run along the field. The average over a group is the Reynolds projection onto the responses
    the group leaves unchanged: the closest such response in the Frobenius norm, still positive. Symmetry
    constrains the response, not its gains.
    """
    # Each principal axis reads the driving field by their pairing and contributes heat flow along
    # itself. Positive gains make this a passive conductivity.
    conductivity = (axes * (Field & axes) * gains).sum(axis=0)                     # [] Direction- <- Field+
    # Pull the field into each rotated frame, apply the conductivity, and turn the flow back. The mean is
    # unchanged by any rotation in the group: those rotations merely permute the terms.
    return Extensor.stack([conductivity] + [(group >> conductivity(group << Field)).mean(axis=0) for group in groups])


cube = cube_rotations()                                                        # [rotations] Rotor+
# Half turns in `mv.yz` and `mv.xy`; quarter turns in `mv.xy`; the rotations of a cube.
groups = [(turns(mv.yz, 2)[:, None] * turns(mv.xy, 2)[None, :]).reshape(-1), turns(mv.xy, 4), cube]
responses = conductivities(axes, gains, groups)                                # [candidate and groups] Direction- <- Field+
fluxes = responses(driving)                                                    # [candidate and groups] Direction-
save_figure(render.draw_conduction(responses, driving, fluxes), "symmetry_conductivity");

In [ ]:
# The unit driving field read on the flow, over the flow's length: the cosine of the angle between them.
deflection = np.degrees(np.arccos(((driving & fluxes) / fluxes.norm()).to_array()))
print("angle between the driving field and the heat flow, candidate and each group:", deflection)
print("the cube's conductivity:\n", responses[-1].coefficient_table())

## 2. Conduction and elasticity of a cubic lattice

A simple-cubic lattice has axial and face-diagonal bonds, directions, and the cube's 24 rotations turn one seed bond of each kind into its whole family. Averaged over them, a bond's conduction, the driving field paired with the bond and returned along it, is the same in every direction, 2/3. Its elasticity, a form on four planes, each reading a bond's stretch, `plane & bond`, is not: the stiffness is 1/2 across the cube's faces and 1/3 across the planes normal to its body diagonals. In the figure, the lattice on the left shows its axial bonds in orange and its diagonal bonds in blue; the middle panel is the conduction, the image of the unit ball of fields, a sphere; the right panel is the stiffness for a strain along every direction, as a radius.

In [ ]:
def lattice_responses(seeds: Direction, cube: Rotor) -> tuple[Conductivity, Elasticity]:
    """The conductivity and the elasticity of a simple-cubic lattice's bonds, averaged over the cube's rotations.

    Every member of a bond family has the same weight, so another cube rotation only permutes the terms in the
    average: this is the microscopic meaning of the group projection.

    The two families carry equal total weights. For the elastic model these are unstressed central springs at
    lattice spacing 1, of axial stiffness 1/3 and face-diagonal stiffness 1/12, whose bond counts and squared
    lengths give each family weight 1; the diagonal springs provide the shear stiffness. This model has equal
    elastic constants c12 and c44, a relation a general cubic crystal need not have.

    A uniaxial strain of size s across a unit plane, `s * plane * (plane & Direction)`, stretches a unit bond by
    the fraction `s * (plane & bond) ** 2`, and the spring energy, its square, makes the elasticity a form on
    four planes. Its value `elasticity(plane, plane, plane, plane)` is the stiffness, with energy density
    `s ** 2 * elasticity(plane, plane, plane, plane) / 2`; it holds transverse strain fixed, and is not the
    relaxed Young's modulus.

    Cubic elasticity background: https://www.ctcms.nist.gov/oof/oof1/Manual/node152.html
    """
    # These are free bond directions; the actual lattice sites are projective points.
    bonds = cube[:, None] >> seeds[None, :]                                      # [rotations, families] Direction-
    # Each bond read by a plane; a uniaxial strain across the plane stretches the bond by the reading's square.
    readings = Plane & bonds                                                     # [rotations, families] Scalar+ <- Plane+

    # Average over the group, then add the two families. No direction is privileged within either
    # orbit. Conductivity reads the field along each bond and returns flow along it.
    conductivity = (bonds * (Field & bonds)).mean(axis=0).sum(axis=0)            # [] Direction- <- Field+
    # Four independent inputs encode the spring stiffness. The same group average
    # leaves cubic structure here, although it made the conduction isotropic.
    elasticity = (readings * readings * readings * readings).mean(axis=0).sum(axis=0)   # [] Scalar+ <- (Plane+, Plane+, Plane+, Plane+)
    return conductivity, elasticity


seeds = mv.vector([[1, 0, 0], [1, 1, 0]]).dual().normalized()                  # [families] Direction-
centre, axial, diagonal, corners = lattice_samples()
conductivity, elasticity = lattice_responses(seeds, cube)
# The unit plane across each direction of the sphere probes a uniaxial strain along it; the stiffness becomes
# a radius.
probes = unit_fields                                                           # [latitudes, longitudes] Plane+
stiffness_surface = sphere * elasticity(probes, probes, probes, probes)        # [latitudes, longitudes] Direction-
save_figure(render.draw_crystal(centre, axial, diagonal, corners, conductivity, stiffness_surface), "symmetry_crystal_lattice");

In [ ]:
cube_face, octahedral_face = mv.x, (mv.x + mv.y + mv.z).normalized()           # [] Plane+, Plane+
print("stiffness along an axis and along a body diagonal:", elasticity(*[cube_face] * 4).to_array(), elasticity(*[octahedral_face] * 4).to_array())

## 3. Optical activity

A rotary power turns light's polarization by an amount with a sense, so it is modelled as an intrinsic, handed scalar, read off the light's direction by the crystal's principal planes, `plane & direction`. A point group holds rotations and, if the crystal has mirrors or a centre of inversion, improper rotations, which reverse that sense. Averaged over the rotations of a square, two rotary powers remain; with the inversion, none. Quarter turns through the inversion leave opposite senses along x and y and none along z: such a crystal is its own mirror image, yet turns light. The cube's rotations leave one rotary power, the same in every direction, as in a sugar solution; with the inversion, none. Each sphere is the sphere of light directions, coloured by the rotary power along each, red one sense and blue the other: the candidate, then the five groups.

In [ ]:
# The planes across the measured responses' principal axes, and the rotary powers across them.
principal_planes = pose >> stack((mv.x, mv.y, mv.z))                           # [axes] Plane+
powers = mv(RotaryPower, [[3.0], [-1.0], [0.5]])                               # [axes] RotaryPower-

In [ ]:
def rotary_powers(planes: Plane, powers: RotaryPower, groups: list[tuple[Rotor, ImproperRotation]]) -> Gyration:
    """A candidate rotary power with principal planes, followed by its average over each point group.

    Light crossing an optically active crystal along a unit direction `d` has its plane of polarization
    turned, in proportion to the distance, by the rotary power `gyration(d, d)`.
    """
    gyration = ((planes & Direction) * (planes & Direction) * powers).sum(axis=0)   # [] RotaryPower- <- (Direction-, Direction-)

    def average(rotations: Rotor, improper: ImproperRotation) -> Gyration:
        # Turn both directions into each element's frame, and carry the rotary power back out: an improper
        # rotation reverses its sense.
        return concatenate([element >> gyration(element << Direction, element << Direction) for element in (rotations, improper)]).mean(axis=0)

    return Extensor.stack([gyration] + [average(rotations, improper) for rotations, improper in groups])


# The inversion is the pseudoscalar as an odd versor.
inversion = mv.xyz                                                             # [] ImproperRotation+
quarter = turns(mv.xy, 4)                                                      # [turns] Rotor+
# Each group's rotations, and its improper rotations; chiral groups have none, an empty batch: the
# quarter turns, with the inversion, through the inversion, the cube's rotations, and with the inversion.
point_groups = [(quarter, inversion * quarter[:0]), (quarter, inversion * quarter), (quarter[::2], inversion * quarter[1::2]),
                (cube, inversion * cube[:0]), (cube, inversion * cube)]
rotary = rotary_powers(principal_planes, powers, point_groups)                 # [candidate and groups] RotaryPower- <- (Direction-, Direction-)
rotary_surfaces = rotary[:, None, None](sphere[None, :, :], sphere[None, :, :])   # [candidate and groups, latitudes, longitudes] RotaryPower-
save_figure(render.draw_optical_activity(rotary), "symmetry_optical_activity");

In [ ]:
print("largest rotary power left, per group:", rotary_surfaces[1:].norm().to_array().max(axis=(1, 2)))

In [ ]:
# checks
# The cube's conductivity is the mean gain in every direction; lattice conduction is isotropic, stiffness
# cubic; the inversion removes every rotary power, and the cube's rotations leave the mean one.
assert (responses[-1](unit_fields) - sphere * gains.mean()).norm().to_array().max() < 1e-7
np.testing.assert_allclose((unit_fields & conductivity(unit_fields)).to_array(), 2 / 3, rtol=1e-7)
np.testing.assert_allclose(elasticity(*[cube_face] * 4).to_array(), 1 / 2, rtol=1e-7)
np.testing.assert_allclose(elasticity(*[octahedral_face] * 4).to_array(), 1 / 3, rtol=1e-7)
for centrosymmetric in (rotary_surfaces[2], rotary_surfaces[5]):
    assert centrosymmetric.norm().to_array().max() < 1e-12
assert (rotary_surfaces[4] - powers.mean(axis=0)).norm().to_array().max() < 1e-12